# Human-AI Collaboration with MOSAIC

MOSAIC is a grid-based **search-and-rescue** game: a building on fire, victims trapped in rooms, lava on the floor, some doors locked.
A human, an AI, or both together must save the victims before time runs out. In this notebook **you play it, right here in the browser**,
while you learn what is inside it: how the agent acts and observes, what you can customize, how an AI teammate helps, and how a play session is
recorded and analysed (your actions and your eye gaze).

## Overview

| Step | What you do |
|---|---|
| 1. The config file and the environment | Build the search-and-rescue task from `config.yaml` |
| 2. Actions | Move the rescuer with `env.step`, let a random agent play with no human, then play the game yourself |
| 3. What the agent observes | Play with only the game view and watch the observation change, then see a snapshot of the level through five cameras |
| 4. The human interface | Look at the participant's window: game view, information panel, chat |
| 5. What you can customize | Change the building, the decoys, the rewards, the lava and the locked doors, and play your own level |
| 6. The AI teammate | See how the two prompt styles differ, plug in a teammate, and ask it for help in the game |
| 7. Record game state and eye gaze | Play again while the game and the gaze are recorded to an XDF file |
| 8. Analyse gaze against the game | Load the file, find where the participant looked and when |

## How to play

Wherever a cell opens a game, **click the picture first**, then use the keys. Press **Stop** to end the game and see your score.

| Key | Does |
|---|---|
| Arrow up | walk forward |
| Arrow left, right | turn |
| `Space` | open a door |
| `E` or `Tab` | pick up a key, or rescue the victim in front of you |
| `Q` or `Alt` | ask the AI teammate |

## Before you start

Select the **Python (smc)** kernel. Keep `config.yaml`, `lsl_tools.py` and `live_play.py` in the same folder as this notebook.

In [ ]:
# Setup: draw pygame off-screen and keep the output quiet. This must run before pygame is imported.
import os, sys, tempfile, warnings, logging
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"
os.environ["PYGAME_HIDE_SUPPORT_PROMPT"] = "1"        # no pygame banner
warnings.filterwarnings("ignore")                     # no warnings
os.environ["PYTHONWARNINGS"] = "ignore"               # ... also in anything this notebook starts
logging.getLogger().setLevel(logging.ERROR)           # ... and no log messages below errors
with tempfile.NamedTemporaryFile("w", suffix=".cfg", delete=False) as lsl_config:
    lsl_config.write("[log]\nlevel = -2\n")            # no log lines from the LSL library
os.environ["LSLAPICFG"] = lsl_config.name
sys.path.insert(0, os.getcwd())

import minigrid.envs.babyai.core.roomgrid_level as roomgrid
roomgrid.print = lambda *args, **kwargs: None         # the level generator prints "Sampling rejected" while it retries a level: these are not errors


: 

## Step 1: The config file and the search-and-rescue environment

MOSAIC is built on **MiniGrid**: its level class extends the **BabyAI** level generator (rooms, doors, keys), and the search-and-rescue task adds victims, lava, cameras and rewards. `build_sar_env` creates the whole task: a building of `num_rows x num_cols` rooms, each
`room_size` tiles wide. Three **placers** decide what goes where:

| Placer | Decides |
|---|---|
| `VictimPlacer(num_real_victims=n)` | how many victims per room, and where |
| `LavaPlacer(lava_per_room=n)` | where the lava hazards are |
| `LockedRoomPlacer(locked_room_prob=p)` | which doors are locked (their keys are hidden in other rooms) |


First the **config file**: a short YAML file next to this notebook. The notebook builds a small building from it, so everything runs in
seconds.

In [ ]:
print(open("config.yaml").read())

In [ ]:
import random

import numpy as np
import matplotlib.pyplot as plt

from mosaic.sar.env import build_sar_env
from mosaic.sar.placers import VictimPlacer, LavaPlacer, LockedRoomPlacer

import yaml

with open("config.yaml") as f:
    config = yaml.safe_load(f)
GAME = config["game"]
print("game config:", GAME)

def make_env(game=None, **changes):
    """Build the task from the ``game`` section of the config (optionally with some values changed)."""
    g = {**(game or GAME), **changes}
    return build_sar_env(
        screen_size=g["screen_size"], num_rows=g["num_rows"], num_cols=g["num_cols"],
        room_size=g["room_size"], tile_size=g["tile_size"],
        victim_placer=VictimPlacer(num_real_victims=g["num_real_victims"]),
        lava_placer=LavaPlacer(lava_per_room=g["lava_per_room"]),
        locked_room_placer=LockedRoomPlacer(locked_room_prob=g["locked_room_prob"]),
    )

def reset_level(env, seed):
    """Start level ``seed``. The placers draw from Python's ``random`` module, so seed it together with the environment."""
    random.seed(seed)
    return env.reset(seed=seed)

env = make_env()
obs, info = reset_level(env, 0)
print("mission:", obs["mission"])
print("max steps:", env.max_steps)


A different `seed` builds a different building. The seed makes experiments **reproducible**: everyone who uses seed 0 plays the same level.
`env.reset(seed=0)` on its own is not enough: it fixes the rooms and doors, but MOSAIC's placers put the victims, lava and keys down with
Python's `random` module. `reset_level` seeds both.

**Checkpoint.** The task parameters (rooms, victims, lava, locks) all come from `config.yaml`; change them there and re-run.

## Step 2: Actions: move the rescuer

The rescuer has seven discrete actions:

| Action | Effect |
|---|---|
| `left`, `right` | turn 90 degrees |
| `forward` | move one tile ahead |
| `pickup` | rescue the victim in front |
| `drop`, `toggle`, `done` | drop an item, open a door, finish |

`env.step(action)` returns the Gymnasium tuple `(observation, reward, terminated, truncated, info)`.

In [ ]:
print(list(env.actions))

reset_level(env, 0)
print("start:", tuple(env.agent_pos), "facing", env.agent_dir)
script = [env.actions.right] + [env.actions.forward] * 5 + [env.actions.left] + [env.actions.forward] * 4
rewards = []
for action in script:
    obs, reward, terminated, truncated, info = env.step(action)
    rewards.append(reward)
print(f"after {len(script)} actions:", tuple(env.agent_pos), "facing", env.agent_dir, " total reward:", sum(rewards), " info:", info)

Rewards come from **events**: a rescued victim is `+1` by default (`RescueRewards` sets the values), a decoy or a dead victim is negative.
The `info["events"]` list says what happened on each step.

### No human needed

MOSAIC follows the Gymnasium API: `env.reset()` starts an episode, and `env.step(action)` returns `(observation, reward, terminated, truncated, info)`.
So the game needs no window and no person. Here a **random agent** plays one whole episode; an agent trained with reinforcement learning follows the same loop, with its own policy in place of `action_space.sample()`.

In [ ]:
reset_level(env, 1)
total_reward = 0.0
for steps in range(1, env.max_steps + 1):
    action = env.action_space.sample()                # a random agent: a learned policy would go here
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    if terminated or truncated:
        break
status = env.get_mission_status()
print(f"random agent: {steps} steps, reward {total_reward:+.1f}, rescued {status['saved_victims']} of {status['saved_victims'] + status['remaining_victims']}")

### Play it yourself

That was a scripted walk. Now you give the actions. Click the picture, then: arrows turn and walk, `Space` opens a door,
`E` (or `Tab`) picks up a key or rescues the victim in front, `Q` (or `Alt`) asks the AI teammate. Press **Stop** to see your score.

In [ ]:
from mosaic.gui.main import SAREnvGUI
from live_play import play_mosaic

play_mosaic(SAREnvGUI(make_env(), config={**GAME, "max_time": 2}))


**Checkpoint.** You played MOSAIC in the browser: every key press was one of the actions above. Next we look at what the agent observes.

## Step 3: What the agent observes

`env.step` and `env.reset` also return an **observation**: a dictionary with everything a program needs to understand the game:
the agent's position and direction, the whole building as a **grid of integer codes**, how many victims are saved and how much
health each victim has left.
Play below: only the game view this time, and the box beside it shows the observation, updated after every action. Press **Stop** when you are done.


In [ ]:
from mosaic.gui.user import User
from live_play import play_view

player = User(make_env())
play_view(player)

The `grid` is a table of codes. `mosaic.sar.observations` defines them: `0` empty floor, `1` wall, `4` lava, `5` real victim, `6` decoy (`FAKE_VICTIM`),
`10`-`29` doors (colour, open/locked), `30`+ keys. After you press **Stop**, here is a snapshot of the level you just played, and the size of its grid.

In [ ]:
obs = player.obs                                  # the observation after your last action

plt.figure(figsize=(4, 4)); plt.imshow(player.env.render()); plt.axis("off"); plt.title("A snapshot of the level you played")
plt.show()
print("grid shape:", np.array(obs["grid"]).shape, "  victim health:", obs["victim_health"])

### The camera decides what the agent sees

The same level can be shown in different ways. A **camera** chooses which part of the building the participant sees, and MOSAIC comes with five.
Here is the level you just played through each of them:

| Camera | The participant sees |
|---|---|
| `FullviewCamera` | the whole building |
| `AgentCenteredCamera` | the agent's room plus a margin |
| `EdgeFollowCamera` | a window that scrolls when the agent nears its edge (the default) |
| `AgentFOVCamera` | only the agent's current room |
| `AgentConeCamera` | the agent's room, with everything outside its forward cone blacked out |

To use one in your own game, pass it as `camera_strategy=` to `build_sar_env`, or switch at any time with `env.switch_camera(...)`.

In [ ]:
from mosaic.core.camera import FullviewCamera, AgentCenteredCamera, EdgeFollowCamera, AgentFOVCamera, AgentConeCamera

cameras = {"FullviewCamera": FullviewCamera(), "AgentCenteredCamera": AgentCenteredCamera(), "EdgeFollowCamera": EdgeFollowCamera(),
           "AgentFOVCamera": AgentFOVCamera(), "AgentConeCamera": AgentConeCamera()}

original = player.env.unwrapped.camera                 # put it back afterwards
fig, axes = plt.subplots(1, len(cameras), figsize=(15, 3.2))
for ax, (title, camera) in zip(axes, cameras.items()):
    player.env.switch_camera(camera)
    ax.imshow(player.env.render()); ax.set_title(title, fontsize=9); ax.axis("off")
player.env.switch_camera(original)
plt.show()

**Checkpoint.** The picture is the level you just played. The `grid` in the observation holds the same level as integer codes: walls around rooms, doors between them, victims where you see them.
`victim_health` is each victim's remaining health.

## Step 4: The human interface

`SAREnvGUI` is the pygame window you just played in: the game view on the left, an information panel (victims saved, time left)
and a chat panel for the AI teammate on the right. `gui.user` holds the game state; `gui.user.step(action)` plays one action,
exactly like a key press.


In [ ]:
from mosaic.gui.main import SAREnvGUI

gui = SAREnvGUI(make_env(), config=GAME)
print("window size:", gui.window_size, "  game view:", gui.game_size, "px, panel width:", gui.panel_width, "px")

## Step 5: What you can customize

Everything about the task is a setting. These are the knobs, and where each one lives:

| What | Setting | Where |
|---|---|---|
| Size of the building | `num_rows`, `num_cols` (rooms down and across), `room_size` (tiles per room side) | `build_sar_env(...)`, `config.yaml` |
| Real victims | `num_real_victims`, per room | `VictimPlacer` |
| **Decoys**: look-alike victims that cost points | `num_decoys`, per room | a placer you write yourself: the next cell does it in a few lines |
| Lava | `lava_per_room` | `LavaPlacer` |
| Locked doors (their keys are hidden in other rooms) | `locked_room_prob` | `LockedRoomPlacer` |
| What the participant sees | a camera: `AgentFOVCamera`, `AgentConeCamera`, `EdgeFollowCamera` | `camera_strategy=` (Step 3 shows all five) |
| **Rewards**: what each outcome is worth | `RescueRewards(fake_victim=-5.0)`: a real victim is `+1` and a decoy `-1` by default | `action=RescueAction(rewards=...)` |
| The clock | `max_time`, in minutes | `config` of `SAREnvGUI` |
| The AI teammate | the client, `prompt_type` (`sparse` or `detailed`), `llm_nudge_interval` | Step 6 |

A **placer** is a small class with one method, `place_all`, that puts objects into the level. Writing your own is the way to change what a level contains.
Real victims, decoys and lava are counted **per room**, so the totals grow with the size of the building, and so does the number of steps the game allows (`max_steps`).

In [ ]:
from mosaic.core.camera import FullviewCamera, AgentFOVCamera, AgentConeCamera
from mosaic.sar import observations as O
from mosaic.sar.actions import RescueAction, RescueRewards
from mosaic.sar.objects import FakeVictim
from mosaic.sar.placers import LavaPlacer

class VictimsAndDecoys(VictimPlacer):
    """Our own placer: the real victims of ``VictimPlacer``, plus look-alike decoys at random free cells."""
    def __init__(self, num_real_victims=2, num_decoys=0):
        super().__init__(num_real_victims)
        self.num_decoys = num_decoys

    def place_all(self, level_gen, num_rows, num_cols):
        super().place_all(level_gen, num_rows, num_cols)                  # the real victims first
        next_to_door = {(x + dx, y + dy) for x, y in self._door_positions(level_gen) for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]}
        for i in range(num_rows):
            for j in range(num_cols):
                room = level_gen.get_room(i, j)
                for _ in range(self.num_decoys):
                    free = self._room_candidates(level_gen, room, next_to_door)
                    if not free:
                        break                                              # the room is full
                    x, y = random.choice(free)
                    level_gen.put_obj(FakeVictim(random.choice(["left", "right"]), random.choice(self.DIRECTIONS)), x, y)

def make_level(rows=2, cols=2, room_size=8, real=2, decoys=0, lava=1, locked=0.35, camera=None, decoy_cost=None, seed=None):
    """A search-and-rescue level. ``real``, ``decoys`` and ``lava`` are counted per room. ``decoy_cost`` is the reward for rescuing a decoy (default -1)."""
    if seed is not None:
        random.seed(seed)                                   # the placers draw from Python's random module
    return build_sar_env(screen_size=640, num_rows=rows, num_cols=cols, room_size=room_size,
                         victim_placer=VictimsAndDecoys(num_real_victims=real, num_decoys=decoys),
                         lava_placer=LavaPlacer(lava_per_room=lava, enabled=lava > 0),
                         locked_room_placer=LockedRoomPlacer(locked_room_prob=locked),
                         camera_strategy=camera,
                         action=RescueAction(rewards=RescueRewards(fake_victim=decoy_cost)) if decoy_cost is not None else None)

Five levels, each changing one setting from the default (seed 0, so you can compare). They are drawn with the whole-building camera, so you can see
every victim, decoy and lava tile:

In [ ]:
variants = {"default": {}, "3 x 3 rooms": dict(rows=3, cols=3), "+3 decoys per room": dict(decoys=3),
            "4 lava per room": dict(lava=4), "90% rooms locked": dict(locked=0.9)}

fig, axes = plt.subplots(1, len(variants), figsize=(16, 3.8))
for ax, (title, changes) in zip(axes, variants.items()):
    env = make_level(**changes, seed=0)
    obs, _ = env.reset(seed=0)
    grid = np.array(obs["grid"])
    env.switch_camera(FullviewCamera())                     # the whole building
    ax.imshow(env.render()); ax.axis("off")
    ax.set_title(f"{title}\n{(grid == O.VICTIM).sum()} victims, {(grid == O.FAKE_VICTIM).sum()} decoys, {(grid == O.LAVA).sum()} lava\n"
                 f"max steps {env.max_steps}", fontsize=9)
plt.tight_layout(); plt.show()

### Build your own level

Change the numbers and run the cell to play the level you made. Decoys look like victims, but rescuing one costs points. Can you tell them apart?
Press **Stop** to see your score: it includes what the decoys cost you.

In [ ]:
ROWS, COLS, ROOM_SIZE = 2, 2, 8        # the building: rooms down, rooms across, tiles per room side
REAL, DECOYS, LAVA = 2, 3, 2           # per room: real victims, decoys, lava tiles
LOCKED = 0.35                          # the chance that a room is locked
DECOY_COST = -5.0                      # the reward for rescuing a decoy (the default is -1; a real victim is +1)
CAMERA = AgentFOVCamera()              # what you see: AgentFOVCamera, AgentConeCamera or EdgeFollowCamera

play_mosaic(SAREnvGUI(make_level(ROWS, COLS, ROOM_SIZE, REAL, DECOYS, LAVA, LOCKED, CAMERA, DECOY_COST), config={**GAME, "max_time": 3}))

## Step 6: The AI teammate

The AI teammate is any object with one method, `query(prompt) -> str`. MOSAIC does the rest:
1. it turns the current observation into a **prompt**: the rules, what the agent can see, and how to answer,
2. it sends the prompt to the client,
3. it cleans the answer into plain speech, and shows it in the chat panel.

There are two prompts, chosen by `prompt_type`. They describe the level in the same way and differ in **how the teammate should answer**:
`sparse` asks for one short radio sentence, `detailed` for a short briefing. In the game, `Q` asks the teammate (`Alt` in the full game), and
`llm_nudge_interval` makes it speak up on its own every so many steps. Here are the answer rules of each prompt for a fresh level:

In [ ]:
from mosaic.llm.client import LLMClient, DummyLLMClient, ask
from mosaic.llm.process_prompts import build_prompt

level = make_level(real=2, decoys=2, seed=0)
obs, _ = level.reset(seed=0)

for prompt_type in ("sparse", "detailed"):
    prompt = build_prompt(obs, prompt_type=prompt_type)
    rules = prompt.split("OUTPUT RULES")[1].split("- The response must contain")[0]
    print(f"--- {prompt_type}: the AI is sent {len(prompt)} characters. Its answer rules:{rules}")

A teammate only has to answer. MOSAIC's placeholder always says the same thing. The stand-in below needs no API key: it reads which prompt it was given and
answers in that style, the way a real model would. `ask` builds the prompt, calls `query`, and cleans the reply (the `<START>` and `<END>` markers the prompt asks for are removed):

In [ ]:
class StandInTeammate(LLMClient):
    """A stand-in for a real model: it answers in the style the prompt asks for."""
    def query(self, prompt):
        if "Maximum 20 words" in prompt:                    # the sparse prompt asks for one short sentence
            return "<START> Survivor to your East, pick them up, then head North. <END>"
        return ("<START> Survivor to your East, they are your best chance right now. "
                "Once you have them, the next one is North past the door. <END>")

print("placeholder:", ask(obs, DummyLLMClient()))
for prompt_type in ("sparse", "detailed"):
    print(f"{prompt_type:11s}:", ask(obs, StandInTeammate(), prompt_type=prompt_type))

### Play with the teammate

Press `Q` to ask the teammate. The answer appears in the chat panel on the right. Switch `PROMPT_TYPE` to `"detailed"` and ask again to hear the difference.

In [ ]:
PROMPT_TYPE = "sparse"                 # or "detailed"

play_mosaic(SAREnvGUI(make_level(real=2, decoys=2, lava=2, camera=AgentFOVCamera()),
                      config={**GAME, "max_time": 3, "prompt_type": PROMPT_TYPE, "llm_nudge_interval": 50},
                      llm_client=StandInTeammate()))

## Step 7: Record game state and eye gaze to XDF

Now you play again, and the game is recorded as you play. The recording has three parts:

1. A **state stream** from the game: one marker per action you take (`agent_x`, `agent_y`, `step_count`, `saved_victims`, `action`, `reward`),
   plus a marker whenever a victim is rescued.
2. A **gaze stream** from the eye tracker. Here it is synthetic: made-up gaze, so no hardware is needed.
3. A **recorder** that writes both to one XDF file.

With a real eye tracker, MOSAIC records the same two things, and Step 8 reads that file too.

### Areas of interest (AOIs)

We split the screen into the **game view**, the **information panel** and the **chat panel**, and record those rectangles in the file,
so the analysis knows where the participant could look.

In [ ]:
import time, threading
import pygame
from lsl_tools import MarkerOutlet, SyntheticGaze, Recorder
from live_play import play_mosaic

XDF_PATH = "mosaic_session.xdf"

gui = SAREnvGUI(make_env(), config=GAME)
user = gui.user
W, H = gui.window_size
AOIS = {"game": [0, 0, gui.game_size, gui.game_size],
        "info": [gui.game_size, 0, gui.panel_width, gui.game_size // 2],
        "chat": [gui.game_size, gui.game_size // 2, gui.panel_width, gui.game_size // 2]}
print("AOIs (x, y, width, height):", AOIS)

markers = MarkerOutlet("MOSAIC-State")
USE_SYNTHETIC = config["eye_tracker"]["source"] == "synthetic"
gaze = SyntheticGaze(width=W, height=H).start() if USE_SYNTHETIC else None      # "live": use the Gaze stream already on the network

def look(x, y):
    """Steer the synthetic gaze (does nothing when a real eye tracker is the source)."""
    if gaze is not None:
        gaze.look_at(x, y)
own = {markers.source_id, gaze.source_id if gaze is not None else "tobii_eye_tracker"}      # this session's two streams, not anyone else's
recorder = Recorder(XDF_PATH, stream_types=("Markers", "Gaze"), source_ids=own).start()
print("recording:", recorder.streams)

While you play, the synthetic gaze **scans around the agent** in the game view, **glances at the information panel right after every rescue**
to check the counter, and looks at the clock every few seconds. That is a stand-in for how a real participant behaves, and it is the pattern
the analysis will look for. Every action you take is written to the state stream as it happens.

In [ ]:
rng = np.random.default_rng(0)
start = time.time()
glance_until = 0.0                                            # the gaze looks at the information panel until this time
gaze_stop = threading.Event()

markers.push("session_start", screen=[W, H], aois=AOIS, synthetic_gaze=USE_SYNTHETIC)

def record_step(info):
    """The game calls this after every action."""
    global glance_until
    flash(info)                                                   # keep the GUI's own reaction, the feedback flash
    obs = user.obs
    markers.push("state", step_count=int(obs["step_count"]), agent_x=int(obs["agent_x"]), agent_y=int(obs["agent_y"]),
                 saved_victims=int(obs["saved_victims"]), remaining_victims=int(obs["remaining_victims"]),
                 action=int(user.last_action), reward=float(user.last_reward))
    for event in info.get("events", []):
        markers.push("rescue" if event.get("type") == "victim_rescued" else "event", **event)
        glance_until = time.time() - start + 0.6                  # check the counter
    if user.total_steps == 20 and pygame.display.get_surface() is not None:
        pygame.image.save(pygame.display.get_surface(), "mosaic_frame.png")      # the background of the gaze plots in Step 8

def steer_gaze():
    next_clock_check, tick = 5.0, 0
    while not gaze_stop.wait(0.1):
        now = time.time() - start
        info = AOIS["info"]
        if now < glance_until:
            look(info[0] + 0.5 * info[2], info[1] + 0.25 * info[3])
        elif now > next_clock_check:
            look(info[0] + 0.5 * info[2], info[1] + 0.8 * info[3])
            if now > next_clock_check + 0.6:
                next_clock_check = now + 5.0
        elif tick % 4 == 0:
            center = AOIS["game"][2] / 2
            look(center + rng.normal(0, 90), center + rng.normal(0, 90))      # scan around the agent
        tick += 1

def finish():
    """Runs once when the game ends: close the recording."""
    gaze_stop.set()
    markers.push("session_end", saved=user.obs["saved_victims"], steps=user.total_steps)
    saved_path = recorder.stop()
    markers.close()
    if gaze is not None:
        gaze.stop()
    return f"saved {saved_path} ({os.path.getsize(saved_path) / 1024:.0f} KB)"

flash = user.on_step
user.on_step = record_step
if gaze is not None:
    threading.Thread(target=steer_gaze, daemon=True).start()
play_mosaic(gui, on_stop=finish)

**Checkpoint.** Play for a minute, then press **Stop**. The cell prints the name of an `.xdf` file with two streams: the game (states and rescues) and the gaze. Run Step 8 after that.

## Step 8: Load the XDF and analyse gaze against the game

Run this after you have pressed **Stop** in Step 7. To analyse a recording from a real session instead, set `XDF_PATH` to that file and `SCREEN` to the size of the screen it was played on.

In [ ]:
import pandas as pd
from lsl_tools import load_streams, gaze_table, marker_table, study_tables, gaze_area, detect_fixations, gaze_heatmap

XDF_PATH = "mosaic_session.xdf"            # <- or the file of a real session
SCREEN = (1920, 1080)                      # <- the size of the screen a real session was played on
streams = load_streams(XDF_PATH)
for s in streams.values():
    span = s["ts"][-1] - s["ts"][0]
    print(f"{s['type']:8s} {s['name']:14s} {len(s['ts']):6d} samples over {span:5.1f} s{'   <-- SYNTHETIC data' if s['synthetic'] else ''}")

if "GameState" in streams:                 # a real session, as MOSAIC records it
    gaze_data, rows = study_tables(streams, SCREEN)
else:
    gaze_data, rows = gaze_table(streams["Gaze"]), marker_table(streams["Markers"])
t0 = min(gaze_data["t"][0], rows[0]["t"])
gt = gaze_data["t"] - t0
for r in rows: r["t"] -= t0

session = next(r for r in rows if r.get("event") == "session_start")
SCREEN, AOIS = tuple(session["screen"]), session["aois"]
states = pd.DataFrame([r for r in rows if r.get("event") == "state"])
rescues = [r for r in rows if r.get("event") == "rescue"]
print(f"{len(states)} game states, {len(rescues)} rescues, gaze valid {gaze_data['valid'].mean():.0%}")
assert len(states) > 0, "This recording has no game states. In Step 7, take at least one action in the game, then press Stop, and run this step again."

### Where did the participant look? Dwell time per area of interest

In [ ]:
def inside(x, y, rect):
    return (x >= rect[0]) & (x < rect[0] + rect[2]) & (y >= rect[1]) & (y < rect[1] + rect[3])

valid = gaze_data["valid"]
x, y = gaze_data["x"], gaze_data["y"]
share = {name: float(inside(x, y, rect)[valid].mean()) for name, rect in AOIS.items()}
print({k: f"{v:.0%}" for k, v in share.items()})

fixations = detect_fixations(gaze_data["t"], x, y, max_dispersion=40, min_duration=0.10)
heat = gaze_heatmap(x, y, shape=(SCREEN[1], SCREEN[0]))
background = plt.imread("mosaic_frame.png") if "Markers" in streams and os.path.exists("mosaic_frame.png") else None

AREA_COLORS = {"game": "#2a78d6", "info": "#eb6834", "chat": "#1baf7a"}     # one colour per area, in every plot below

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].bar(share.keys(), share.values(), width=0.5, color=[AREA_COLORS[name] for name in share]); axes[0].set_ylabel("share of valid gaze"); axes[0].set_title("Dwell time by area")
for ax in axes[1:]:
    if background is not None:
        ax.imshow(background, extent=[0, SCREEN[0], SCREEN[1], 0])
    else:                                  # no screenshot of that session: outline the areas instead
        for rect in AOIS.values():
            ax.add_patch(plt.Rectangle(rect[:2], rect[2], rect[3], fill=False, edgecolor="gray"))
        ax.set_aspect("equal")
    ax.set_xlim(0, SCREEN[0]); ax.set_ylim(SCREEN[1], 0); ax.axis("off")
axes[1].imshow(heat, extent=[0, SCREEN[0], SCREEN[1], 0], cmap="hot", alpha=0.55); axes[1].set_title("Gaze heatmap")
for f in fixations:
    axes[2].scatter(f["x"], f["y"], s=(f["end"] - f["start"]) * 3000, facecolors="none", edgecolors="cyan", linewidths=1.5)
axes[2].set_title(f"{len(fixations)} fixations")
plt.tight_layout(); plt.show()

### Does gaze follow the game events?

After each rescue, does the participant check the information panel? We compare the share of gaze on the panel in the **1.5 s after a rescue**
with the share in all **other** times. The strip under the progress line shows which area the gaze was in at every moment.

In [ ]:
on_info = inside(x, y, AOIS["info"]) & valid
after = np.zeros(len(gt), dtype=bool)
for r in rescues:
    after |= (gt >= r["t"]) & (gt < r["t"] + 1.5)

if rescues:
    print(f"gaze on the info panel within 1.5 s after a rescue: {on_info[after].mean():.0%}")
    print(f"gaze on the info panel at other times:              {on_info[~after].mean():.0%}")
else:
    print("No victim was rescued in this session, so there is nothing to compare.")

area = gaze_area(gaze_data, AOIS)             # per gaze sample: "game", "info", "chat", "elsewhere" or "no data"

fig, axes = plt.subplots(2, 1, figsize=(13, 5), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
axes[0].plot(states["t"], states["saved_victims"], drawstyle="steps-post", color="#52514e", linewidth=2)
for r in rescues:
    for ax in axes: ax.axvline(r["t"], color="#898781", linewidth=1)
axes[0].set_ylabel("victims saved"); axes[0].yaxis.get_major_locator().set_params(integer=True); axes[0].set_title("Game progress (vertical lines = rescues) and the area the eyes were in")
for name, color in {**AREA_COLORS, "elsewhere": "#c3c2b7"}.items():
    axes[1].fill_between(gt, 0, 1, where=(area == name), step="post", color=color, linewidth=0, label=name)
axes[1].set_yticks([]); axes[1].set_xlabel("time (s)")
axes[1].legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.55), frameon=False, title="gaze area (blank = no valid gaze)")
plt.tight_layout(); plt.show()

### The rescuer's path through the building

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.scatter(states["agent_x"], states["agent_y"], c=states["t"], cmap="viridis", s=18)
for r in rescues:
    nearest = states.iloc[(states["t"] - r["t"]).abs().argmin()]
    ax.plot(nearest["agent_x"], nearest["agent_y"], "r*", markersize=16)
ax.invert_yaxis(); ax.set_aspect("equal"); ax.set_title("Path (colour = time), red stars = rescues")
plt.show()

## Summary

| What we did | Tool |
|---|---|
| Built a search-and-rescue task from a YAML config and looked at it as an image, a grid and numbers | `config.yaml`, `build_sar_env`, MiniGrid / Gymnasium |
| Drew the participant's screen off-screen | `SAREnvGUI` on the pygame dummy driver |
| Played the game live in the browser | `SAREnvGUI.handle_user_input` |
| Streamed game state and gaze with LSL and saved them as XDF | `pylsl`, `lsl_tools.Recorder` |
| Linked gaze to game events | `pyxdf`, areas of interest, fixations |
| Read the recording of a real session with an eye tracker | `lsl_tools.study_tables` |

The same four steps fit any human-AI study: **simulate, interact, record, analyse.**